In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS 03_global_tech_gold.facts_dims_tables;

In [0]:
from pyspark.sql.functions import lit, when, upper, trim, col, date_format, year, month, quarter, dayofmonth, dayofweek, weekofyear, sequence, explode, coalesce


In [0]:
accounts_df = spark.table("02_global_tech_silver.transformed_tables.transformed_accounts")
pay_df = spark.table("02_global_tech_silver.transformed_tables.transformed_payroll")
gl_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_general_ledgers")
emp_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_employee")
dept_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_departments")
company_df = spark.table("02_global_tech_silver.transformed_tables.transformed_company")


In [0]:
# Date Dimension
min_gl_date = gl_df.agg({"entry_date": "min"}).collect()[0][0]
max_gl_date = gl_df.agg({"entry_date": "max"}).collect()[0][0]
min_pay_date = pay_df.agg({"pay_date": "min"}).collect()[0][0]
max_pay_date = pay_df.agg({"pay_date": "max"}).collect()[0][0]

all_dates = [min_gl_date, max_gl_date, min_pay_date, max_pay_date]
start_date = min([d for d in all_dates if d is not None])
end_date   = max([d for d in all_dates if d is not None])

dim_date = (
    spark.createDataFrame([(start_date, end_date)], ["start", "end"])
        .withColumn("date", explode(sequence(col("start"), col("end"))))
        .drop("start", "end")
        .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
        .withColumn("year", year(col("date")))
        .withColumn("month", month(col("date")))
        .withColumn("quarter", quarter(col("date")))
        .withColumn("month_name", date_format(col("date"), "MMMM"))
        .withColumn("year_month", date_format(col("date"), "yyyy-MM"))
        .withColumn("day_of_month", dayofmonth(col("date")))
        .withColumn("day_of_week", dayofweek(col("date")))
        .withColumn("week_of_year", weekofyear(col("date")))
)

display(dim_date.limit(5))
print(dim_date.count())

In [0]:
# Company dimension
dim_company = company_df.select(
    col("company_sk").alias("company_key"),
    col("company_id"),
    col("company_name"),
    col("industry"),
    col("country"),
    col("established_date"),
    col("company_is_active")
)

In [0]:
# Department dimension

dim_department = dept_df.select(
    col("department_sk").alias("department_key"),
    col("department_id"),
    col("department_code"),
    col("department_name"),
    col("cost_center")
)

In [0]:
# Employee dimension

dim_employee = emp_df.select(
    col("employee_sk").alias("employee_key"),
    col("employee_id"),
    col("employee_code"),
    col("first_name"),
    col("last_name"),
    col("email"),
    col("department_id"),
    col("company_id"),
    col("position"),
    col("hire_date"),
    col("termination_date"),
    col("base_salary"),
    col("employee_is_active"),
    col("tenure_days")
)

In [0]:
# Accounts Dimension with revenue/expense categorization
dim_account = accounts_df.select(
    col("account_sk").alias("account_key"),
    col("account_id"),
    col("account_code"),
    col("account_name"),
    col("account_type"),
    col("category"),
    col("account_is_active")
).withColumn(
    "account_category",
    when(upper(col("category")).contains("REVENUE"), "Revenue")
    .when(upper(col("category")).contains("COST"), "Cost of Sales")
    .when(upper(col("category")).contains("EXPENSE"), "Operating Expense")
    .otherwise("Other")
)

In [0]:
# Financial transactions fact table
fact_general_ledger = gl_df.alias("gl") \
    .join(dim_date.alias("d"), col("gl.posting_date") == col("d.date"), "left") \
    .join(dim_account.alias("a"), col("gl.account_id") == col("a.account_id"), "left") \
    .select(
        col("gl.gl_sk").alias("gl_key"),
        col("gl.gl_id"),
        col("d.date_key"),
        col("gl.entry_date"),
        col("gl.posting_date"),
        col("gl.fiscal_year"),
        col("gl.fiscal_period"),
        col("d.year_month"),
        col("gl.company_id"),
        col("a.account_key"),
        col("gl.account_id"),
        col("a.account_category"),
        col("gl.department_id"),
        col("gl.transaction_type"),
        col("gl.reference_number"),
        col("gl.description"),
        col("gl.debit_amount"),
        col("gl.credit_amount"),
        col("gl.net_amount"),
        col("gl.created_by"),
        col("gl.approved_by"),
        col("gl.gl_status")
    )\
    .withColumn("revenue_amount", when(col("account_category") == "Revenue", col("net_amount")).otherwise(0)) \
    .withColumn("cogs_amount", when(col("account_category") == "Cost of Sales", col("net_amount")).otherwise(0)) \
    .withColumn("opex_amount", when(col("account_category") == "Operating Expense", col("net_amount")).otherwise(0))


In [0]:
# Payroll Fact Table
fact_payroll = pay_df.alias("p") \
    .join(dim_date.alias("d"), col("p.pay_date") == col("d.date"), "left") \
    .join(dim_employee.alias("e"), col("p.employee_id") == col("e.employee_id"), "left") \
    .select(
        col("p.payroll_sk").alias("payroll_key"),
        col("p.payroll_id"),
        col("d.date_key"),
        col("p.pay_date"),
        col("d.year_month"),
        col("e.employee_key"),
        col("p.employee_id"),
        col("e.position"),
        col("p.company_id"),
        col("p.department_id"),
        col("p.pay_period_start"),
        col("p.pay_period_end"),
        col("p.bonus"),
        col("p.overtime_pay"),
        col("p.commission"),
        col("p.allowances"),
        col("p.tax_deduction"),
        col("p.social_security"),
        col("p.health_insurance"),
        col("p.retirement_contribution"),
        col("p.other_deductions"),
        col("p.total_deduction"),
        col("p.total_compensation"),
        col("p.payment_method"),
        col("p.payroll_status"),
        col("e.base_salary")
    ).withColumn(
        "overtime_to_base_ratio",
        when(col("base_salary") > 0, col("overtime_pay") / col("base_salary")).otherwise(0)
    ).withColumn(
        "variable_compensation",
        coalesce(col("bonus"), lit(0)) + coalesce(col("overtime_pay"), lit(0)) + coalesce(col("commission"), lit(0))
    )

print(f"fact_payroll created with {fact_payroll.count()} rows")
display(fact_payroll.limit(5))

In [0]:
from pyspark.sql.functions import last_day, add_months, explode, sequence, to_date, concat_ws


monthly_dates = dim_date.select("year_month").distinct() \
    .withColumn("snapshot_date", to_date(concat_ws("-", col("year_month"), lit("01"))))

employee_months = emp_df.crossJoin(monthly_dates.alias("m"))

fact_employee_monthly_snapshot = employee_months \
    .filter(
        (col("snapshot_date") >= col("hire_date")) &
        ((col("termination_date").isNull()) | (col("snapshot_date") <= col("termination_date")))
    ) \
    .join(dim_date.alias("d"), col("snapshot_date") == col("d.date"), "left") \
    .select(
        col("employee_sk").alias("employee_key"),
        col("employee_id"),
        col("m.year_month"),
        col("d.date_key").alias("snapshot_date_key"),
        col("snapshot_date"),
        col("company_id"),
        col("department_id"),
        col("position"),
        col("base_salary"),
        col("employee_is_active"),
        lit(1).alias("headcount")
    )

print(f"fact_employee_monthly_snapshot created with {fact_employee_monthly_snapshot.count()} rows")
display(fact_employee_monthly_snapshot.limit(10))

In [0]:
# dimension tables
dim_date.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_dims_tables.dim_date")
dim_company.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_dims_tables.dim_company")
dim_department.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_dims_tables.dim_department")
dim_employee.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_dims_tables.dim_employee")
dim_account.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_dims_tables.dim_account")

# fact tables
fact_general_ledger.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_dims_tables.fact_general_ledger")
fact_payroll.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_dims_tables.fact_payroll")
fact_employee_monthly_snapshot.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_dims_tables.fact_employee_monthly_snapshot")
